### importowanie bibliotek

In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
import pandas as pd
from sklearn import metrics
import config_test
import numpy as np

### wybór urządzenia

In [2]:
device = "cpu" # PyTorch pozwala na wybranie urządzenia, na którym będą przeprowadzane operacje
print(f"Using {device} device")

Using cpu device


### sieć neuronowa

In [ ]:
class NeuralNetwork(nn.Module): 
    def __init__(self):
        super().__init__()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(784, 512), # wastwa wejściowa (784 neurony) -> warstwa ukryta (512 neuronów)
            nn.ReLU(), 
            nn.Linear(512, 512), # warstwa ukryta (512 neuronów) -> warstwa ukryta (512 neuronów)
            nn.ReLU(),
            nn.Linear(512, 10), # warstwa ukryta (512 neuronów) -> wartswa wyjściowa (10 neuronów) 
        )

    def forward(self, x): # przesuwamy się z warstwy do warstwy
        logits = self.linear_relu_stack(x)
        return logits

### dostosowanie danych, trenowanie i testowanie modelu

In [ ]:
df = pd.read_csv(config_test.NEW_DATA_PATH) #jednym z lepszych sposobów na wrzucenie danych do tensorów jest użycie dataframe

# Przechodzimy po kolei po wszystkich foldach
for fold in range(config_test.FOLDS_COUNT):
    

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~inicjalizacja danych~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    torch_model = NeuralNetwork().to(device) # model
    
    # dzielimy nasze dane na treningowe i testowe
    df_train = df.loc[df['kfold'] != fold] # dane treningowe = dane, które nie należą do obecnego folda
    df_test = df.loc[df['kfold'] == fold] # dane testowe = dane, które należą do obecnego folda

    # wypisujemy liczbę wierszy i kolumn
    # liczba wierszy powinna wynosić łącznie 10000, bo tyle mamy obrazków
    # liczba kolumn powinna wynosić łącznie 28*28 + 2 (liczba pikseli jednego obrazka + kolumna label i kolumna kfold)
    #print(df_train.shape)
    #print(df_test.shape)

    # zamieniamy dataframe na tensory
    dt_train = torch.tensor(df_train.values, dtype=torch.float32)
    dt_test = torch.tensor(df_test.values, dtype=torch.float32)
    
    # dzielimy dane na dane bez labeli i labele
    # żeby przeprowadzanie operacji na tensorach było możliwe, to dane bez labeli muszą być zmiennoprzecinkowe, a labele całkowite
    dt_train_label = dt_train[:,-2:-1]
    dt_train = dt_train[:,:-2] # tensor treningowy zawierający 28*28 kolumn
    dt_train_label = dt_train_label.long() # tensor treningowy zawierający kolumnę label
    #print(dt_train.shape)
    #print(dt_train_label.shape)

    dt_test_label = dt_test[:,-2:-1]
    dt_test = dt_test[:,:-2] # tensor testowy zawierający 28*28 kolumn
    dt_test_label = dt_test_label.long() # tensor testowy zawierający kolumnę label
    #print(dt_test.shape)
    #print(dt_test_label.shape)
    
    # DataLoader ułatwia dostęp do danych, ładuje dane w określonych partiach (batchach) i losowo miesza dane w każdej epoce
    train_dataset = TensorDataset(dt_train, dt_train_label) # Dataset łączy dane dla DataLoadera
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

    test_dataset = TensorDataset(dt_test, dt_test_label)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

    # przenosimy tensory do urządzenia
    dt_train, dt_train_label = dt_train.to(device), dt_train_label.to(device)
    dt_test, dt_test_label = dt_test.to(device), dt_test_label.to(device)

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~koniec inicjalizacji danych~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~trenowanie danych~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    criterion = nn.CrossEntropyLoss()  # Funkcja kosztu dla klasyfikacji
    optimizer = torch.optim.Adam(torch_model.parameters(), lr=0.001)  # Dostosowuje parametry modelu

    # ustawiamy ziarno generatora liczb losowych, losowe funkcje w PyTorch będą się zachowywać deterministycznie
    torch.manual_seed(42)

    # liczba epok
    epochs = 100

    # pętla treningowa
    for epoch in range(epochs):

        # ustawienie modelu w tryb treningowy
        torch_model.train()

        # oblicza straty, które powstały w trakcie trenowania modelu
        running_loss = 0.0

        for X_batch, y_batch in train_loader: #X_batch - batch z dt_train, y_batch - batch z dt_train_label
            logits = torch_model(X_batch) # przekazujemy batch danych z dt_train do modelu i otrzymujemy logity
            y_batch = y_batch.squeeze(1) # usuwamy dimensions o rozmiarze 1 (konieczne dla następnej funkcji)
            loss = criterion(logits, y_batch) # obliczamy stratę pomiędzy przewidywaniami modelu a labels

            # czyszczenie poprzednich gradientów
            optimizer.zero_grad()

            # backpropagation
            loss.backward()

            # aktualizacja wag i biases
            optimizer.step()

            # do strat dodajemy wartość obliczoną przez criterion
            running_loss += loss.item()
        

        # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~testowanie danych ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

        # ustawienie modelu w tryb oceny
        torch_model.eval()
        correct = 0 # poprawnie odgadnięte cyfry
        total = 0 # wszystkie cyfry
        val_loss = 0 # wartość straty

        with torch.no_grad(): # wyłączenie liczenia gradientów
            for X_val, y_val in test_loader: # X_val - batch z dt_test, y_val - batch z dt_test_label 
                logits = torch_model(X_val) # przekazujemy batch danych z dt_test do modelu i otrzymujemy logity
                y_val = y_val.squeeze(1) # usuwamy dimensions o rozmiarze 1 (konieczne dla następnej funkcji)
                loss = criterion(logits, y_val) # obliczamy stratę pomiędzy przewidywaniami modelu a labels
                val_loss += loss.item() # do strat dodajemy wartość obliczoną przez criterion

                pred = logits.argmax(dim=1) # uzyskujemy indeks klasy przewidywanej
                correct += (pred == y_val).sum().item() # sumujemy liczbę poprawnych przewidywań
                total += y_val.size(0) #zwiększamy całkowitą liczbę przykładów

        accuracy = correct / total

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}, Accuracy: {accuracy}")



Epoch 1/100, Loss: 1.4838377264738083, Accuracy: 0.9225
Epoch 2/100, Loss: 0.15871905064582825, Accuracy: 0.9335
Epoch 3/100, Loss: 0.11223995557427406, Accuracy: 0.936
Epoch 4/100, Loss: 0.06915312788449228, Accuracy: 0.9485
Epoch 5/100, Loss: 0.06711290700547397, Accuracy: 0.941
Epoch 6/100, Loss: 0.11502999766357243, Accuracy: 0.941
Epoch 7/100, Loss: 0.10154620076529682, Accuracy: 0.94
Epoch 8/100, Loss: 0.06105833383603022, Accuracy: 0.9535
Epoch 9/100, Loss: 0.03477574471227126, Accuracy: 0.9475
Epoch 10/100, Loss: 0.04568838184222113, Accuracy: 0.947
Epoch 11/100, Loss: 0.08143729286640883, Accuracy: 0.946
Epoch 12/100, Loss: 0.0932050565246027, Accuracy: 0.933
Epoch 13/100, Loss: 0.04647147741541267, Accuracy: 0.9565
Epoch 14/100, Loss: 0.0889935370386811, Accuracy: 0.9395
Epoch 15/100, Loss: 0.061055377009965016, Accuracy: 0.949
Epoch 16/100, Loss: 0.042367728622703, Accuracy: 0.953
Epoch 17/100, Loss: 0.03019102565383946, Accuracy: 0.9395
Epoch 18/100, Loss: 0.066818993153036